In [1]:
import pandas as pd
import os
from pathlib import Path    

Path to the train tand test datasets

In [2]:
current_path = Path.cwd()
new_path = current_path.parent.parent
os.chdir(new_path)

In [3]:
print(os.getcwd())

/home/piotr/credit-data-exepriments/credit-data


In [4]:
input_filepath_train = Path('data/external/loan-default-prediction/train_v2.csv')
input_filepath_test = Path('data/external/loan-default-prediction/test_v2.csv')

In [5]:
df_train = pd.read_csv(input_filepath_train, index_col=0, header=0, low_memory=False)
df_test = pd.read_csv(input_filepath_test, index_col=0, header=0, low_memory=False)

From the previous analysis not presented in this workbook I know that variables/columns f85 and f86 in the training dataset contain the same values. I will calculate abs diffrence beteen them in train dataset to check it. Later on all variables need to be checked.

In [12]:
ser_diff_train = (df_train['f85'] - df_train['f86']).abs()
ser_diff_test = (df_test['f85'] - df_test['f86']).abs()
print(ser_diff_train.sum(), ser_diff_test.sum())

0.0 43866605.0


In [13]:
counts = ser_diff_test.apply(lambda x: 0 if x == 0 else 1).value_counts(normalize=True)
print(counts)

0    0.500152
1    0.499848
Name: proportion, dtype: float64


It means that half of the observations have the same properties as in the train dataset

In [14]:
df_test[ser_diff_test == 0]

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f769,f770,f771,f772,f773,f774,f775,f776,f777,f778
id,,,,,,,,,,,,,,,,,,,,,
105473,156,6,0.728518,5400,3.0,79754,1455.0,4803.0,153.95,155.50,...,-13.26,24,9.53,-7.55,6.22,0.3030,0.6087,0,1,36
105474,132,9,0.898133,2200,16.0,113,5735.0,2387.0,130.61,131.96,...,-4.99,9,3.25,-2.33,1.69,0.2317,0.4184,0,0,393
105478,152,8,0.946007,3600,2.0,83957,3341.0,1748.0,151.25,151.97,...,-0.93,2,0.73,-0.61,0.52,0.3857,0.3775,1,1,15
105479,133,10,0.463593,2400,16.0,2572,7372.0,9550.0,131.00,133.00,...,-1.18,2,0.74,-0.50,0.35,0.1480,-1.2180,1,0,1079
105480,117,10,0.063169,1800,2.0,81166,72.0,326.0,119.95,117.30,...,-1.11,7,0.28,-0.08,0.03,0.1219,-0.5918,0,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
316409,116,9,0.056028,1500,3.0,82231,48.0,2292.0,120.25,116.79,...,-34.74,67,23.01,-17.02,13.19,0.2731,0.4087,0,0,30
316412,154,9,0.477440,3600,3.0,80502,4283.0,280.0,154.13,153.84,...,-9.64,20,5.98,-4.13,3.02,0.2582,0.4792,1,1,41
316413,150,8,0.480407,4400,7.0,14507,78.0,1502.0,150.75,150.20,...,-1.12,4,0.41,-0.16,0.08,0.1552,0.8375,1,1,20


Dataset was downloaded from https://www.kaggle.com/competitions/loan-default-prediction/data. The only difference between training and test dataset is that 'The competition sponsor has worked to remove time-dimensionality from the data. However, the observations are still listed in order from old to new in the training set. In the test set they are in random order.'

In [9]:
def sum_of_abs_difference(df, col1, col2, n_obs=10_000):
    df_head = df.head(n_obs)
    ser_diff = (df_head[col1] - df_head[col2]).abs()
    return ser_diff.sum()

print(sum_of_abs_difference(df_train, 'f85', 'f86'))
print(sum_of_abs_difference(df_test, 'f85', 'f86'))

print(sum_of_abs_difference(df_train, 'f85', 'f86'))
print(sum_of_abs_difference(df_test, 'f97', 'f98'))

0.0
2156503.0
0.0
1880529.0


In [15]:
def share_of_zero_difference(df, col1, col2):
    ser_diff = (df[col1].fillna(0) - df[col2].fillna(0)).abs()
    return ser_diff.apply(lambda x: 0 if x == 0 else 1).value_counts(normalize=True)

print(share_of_zero_difference(df_test, 'f85', 'f86'))
print(share_of_zero_difference(df_test, 'f97', 'f98'))
print(share_of_zero_difference(df_test, 'f255', 'f256'))

0    0.501071
1    0.498929
Name: proportion, dtype: float64
0    0.501014
1    0.498986
Name: proportion, dtype: float64
0    0.500559
1    0.499441
Name: proportion, dtype: float64


In [16]:
def indices_of_zero_difference(df, col1, col2):
    ser_diff = (df[col1].fillna(0) - df[col2].fillna(0)).abs()
    return ser_diff[ser_diff == 0].index

print(indices_of_zero_difference(df_test, 'f97', 'f98'))
print(indices_of_zero_difference(df_test, 'f255', 'f256'))

Index([105473, 105474, 105478, 105479, 105480, 105481, 105484, 105485, 105487,
       105490,
       ...
       316399, 316400, 316401, 316405, 316407, 316409, 316412, 316413, 316414,
       316415],
      dtype='int64', name='id', length=105686)
Index([105473, 105474, 105478, 105479, 105480, 105481, 105484, 105485, 105487,
       105490,
       ...
       316399, 316400, 316401, 316405, 316407, 316409, 316412, 316413, 316414,
       316415],
      dtype='int64', name='id', length=105590)


In [17]:
def is_weakly_ascending(series):
    """Checks if a Pandas Series has weakly ascending values.

    Args:
        series: A Pandas Series.

    Returns:
        True if the series is weakly ascending, False otherwise.
    """
    return all(series.iloc[i] <= series.iloc[i + 1] for i in range(len(series) - 1))

def detect_weakly_ascending_columns(df):
    """Detects columns in a DataFrame with weakly ascending values.

    Args:
        df: A Pandas DataFrame.

    Returns:
        A list of column names that are weakly ascending.
    """
    ascending_columns = []
    for col in df.columns:
        if is_weakly_ascending(df[col]):
            ascending_columns.append(col)
    return ascending_columns

Check how many columns can be considered weakly ascending in the training and test datasets.

In [18]:
df_train.head(1000).apply(is_weakly_ascending).sum()

np.int64(13)

In [19]:
ascending_columns = detect_weakly_ascending_columns(df_train)
print(ascending_columns)

['f33', 'f34', 'f35', 'f37', 'f38', 'f700', 'f701', 'f702', 'f736', 'f764']


In [20]:
df_train[ascending_columns].tail()

,f33,f34,f35,f37,f38,f700,f701,f702,f736,f764
id,,,,,,,,,,
105467,0,0,0,0,0,0,0,0,1,1
105468,0,0,0,0,0,0,0,0,1,1
105469,0,0,0,0,0,0,0,0,1,1
105470,0,0,0,0,0,0,0,0,1,1
105471,0,0,0,0,0,0,0,0,1,1


In [21]:
df_train[ascending_columns].head()

,f33,f34,f35,f37,f38,f700,f701,f702,f736,f764
id,,,,,,,,,,
1,0,0,0,0,0,0,0,0,1,1
2,0,0,0,0,0,0,0,0,1,1
3,0,0,0,0,0,0,0,0,1,1
4,0,0,0,0,0,0,0,0,1,1
5,0,0,0,0,0,0,0,0,1,1


In [22]:
print(df_train['f3'].std(), df_test['f3'].std())
print(df_train['f5'].std(), df_test['f5'].std())

0.28875240597001955 0.2894584618570656
5.151111736060021 5.4024069581800305


In [23]:
print(df_train['f3'].mean(), df_test['f3'].mean())  

0.4990662571074324 0.5012064065712416


In [24]:
print(df_train['f3'].std(), df_test['f3'].std())
print(df_train['f5'].std(), df_test['f5'].std())

0.28875240597001955 0.2894584618570656
5.151111736060021 5.4024069581800305


In [25]:
print(df_train['f6'].mean(), df_test['f6'].mean())
print(df_train['f9'].mean(), df_test['f9'].mean())

47993.70431682643 53153.42558214503
134.55522503816218 135.1443987029733


In [26]:
import numpy as np
from scipy import stats

In [27]:
def compare_distributions(population_df, train_df, test_df, alpha=0.05):
    """
    Compares the distributions of variables in population, train, and test datasets.

    Args:
        population_df: Pandas DataFrame representing the population data.
        train_df: Pandas DataFrame representing the training data.
        test_df: Pandas DataFrame representing the test data.
        alpha: Significance level for the statistical test (default: 0.05).

    Returns:
        A dictionary containing the mean, std, p-value, and a boolean indicating if
        we fail to reject the null hypothesis (means are considered the same).
    """

    results = {}
    for col in train_df.columns:
        if col in population_df.columns and col in test_df.columns:
            results[col] = {}

            results[col]['population_mean'] = population_df[col].mean()
            results[col]['population_std'] = population_df[col].std()
            results[col]['train_mean'] = train_df[col].mean()
            results[col]['train_std'] = train_df[col].std()
            results[col]['test_mean'] = test_df[col].mean()
            results[col]['test_std'] = test_df[col].std()

            try:
                train_data = train_df[col].dropna()
                test_data = test_df[col].dropna()

                if len(train_data) > 1 and len(test_data) > 1:
                    t_stat, p_value = stats.ttest_ind(train_data, test_data, equal_var=True)
                    results[col]['p_value'] = p_value
                    results[col]['same_means'] = p_value > alpha  # Corrected logic
                else:
                    results[col]['p_value'] = np.nan
                    results[col]['same_means'] = False
                    print(f"Warning: Insufficient data for t-test in column '{col}'.")

            except Exception as e:
                results[col]['p_value'] = np.nan
                results[col]['same_means'] = False
                print(f"Error performing t-test for column '{col}': {e}")
        else:
            print(f"Warning: Column '{col}' not found in all DataFrames.")

    return results

In [28]:
df_combined = pd.concat([df_train, df_test], axis=0)

In [29]:
lst_cols_all = [x for x in df_combined.columns if x != 'loss']
lst_cols_obj = df_combined.dtypes[df_combined.dtypes == 'object'].index.to_list()
lst_cols_not_obj = [x for x in lst_cols_all if x not in lst_cols_obj]

In [30]:
df_results = compare_distributions(df_combined[lst_cols_not_obj], df_train[lst_cols_not_obj], df_test[lst_cols_not_obj])

/home/piotr/credit-data-exepriments/credit-data/.venv/lib/python3.11/site-packages/scipy/stats/_axis_nan_policy.py:573: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


In [31]:
pd.DataFrame(df_results).T.to_excel('data/interim/loan-default-prediction/distribution_comparison.xlsx')

In [32]:
print(df_train['f1'].mean())
print(df_test['f1'].mean())

train_data = df_train['f1'].dropna()
test_data = df_test['f1'].dropna()

stats.ttest_ind(train_data, test_data, equal_var=True)    

print(train_data.shape, test_data.shape)
test_data.shape[0] //2

134.60317053976922
135.12508533070388
(105471,) (210944,)


105472

In [33]:
print(df_train['f85'].sum())
print(df_test['f85'].sum())


33267437.0
77136882.0


In [34]:
df_test.loc[df_test['f138'] == '231785465849730007040', ['f85', 'f86']].sum()

f85    247.0
f86    257.0
dtype: float64

In [35]:
df_test['f85'].sum() == df_test['f86'].sum()

np.True_

In [36]:
df_test_1 = df_test.loc[df_test['f85'] == df_test['f86'],:].copy()
df_test_2 = df_test.loc[df_test['f85'] != df_test['f86'],:].copy()

In [37]:
print(df_test_1['f85'].sum() == df_test_1['f86'].sum())
print(df_test_2['f85'].sum() == df_test_2['f86'].sum())

True
True


In [38]:
# get columns with unique values in df_test_1 but not unique in df_test_2
lst_cols_unique_test_1 = []
lst_cols_unique_test_2 = []
for col in df_test_1.columns:
    if df_test_1[col].nunique() == df_test_1.shape[0]:
        lst_cols_unique_test_1.append(col)
    if df_test_2[col].nunique() == df_test_2.shape[0]:
        lst_cols_unique_test_2.append(col)
        


In [39]:
lst_cols_unique_test_1_not_2 = [x for x in lst_cols_unique_test_1 if x not in lst_cols_unique_test_2]
lst_cols_unique_test_2_not_1 = [x for x in lst_cols_unique_test_2 if x not in lst_cols_unique_test_1]
print(lst_cols_unique_test_1_not_2)
print(lst_cols_unique_test_2_not_1)

[]
[]


In [40]:
df_test_2.iloc[:1][['f85', 'f86']].sum()

f85    842.0
f86     50.0
dtype: float64

In [41]:
df_test_3 = df_test_2[['f85','f86']].copy()
row_iloc = 1
row_index = int(df_test_3.index[row_iloc])
row_to_add = df_test_3.iloc[row_iloc,:]
remaining_rows = df_test_3.drop(row_index, axis=0)
new_df = remaining_rows.add(row_to_add)

In [42]:
print(df_test_3.shape, remaining_rows.shape)

(105440, 2) (105439, 2)


In [43]:
new_df.loc[new_df['f85'] == new_df['f86'],:]

,f85,f86
id,,
134365,1579.0,1579.0
186094,1514.0,1514.0


In [45]:
lst_indices = [row_index] + list(new_df.loc[new_df['f85'] == new_df['f86'],:].index)[:1]
print(lst_indices)

[105475, 134365]


In [46]:
df_test_2.loc[lst_indices,['f85','f86','f87','f88']].sum()

f85    1579.0
f86    1579.0
f87     654.0
f88     604.0
dtype: float64

In [47]:
new_df.loc[[134365], ['f85','f86']].sum()

f85    1579.0
f86    1579.0
dtype: float64

In [48]:
df_test_2['f732'].value_counts()

f732
830768643      747
805893138      301
724890128      267
740516128      265
756142128      170
              ... 
18128835343      1
2921571702       1
7896883581       1
3159828375       1
4087840487       1
Name: count, Length: 92926, dtype: int64

In [49]:
df_test_2.loc[df_test_2['f732'] == 830768643, 'f85':'f86'].sum()

f85    267645.0
f86    273796.0
dtype: float64

In [50]:
df_test_2.loc[:,['f85','f86']]

,f85,f86
id,,
105472,842.0,50.0
105475,1396.0,1.0
105476,153.0,375.0
105477,77.0,87.0
105482,684.0,213.0
...,...,...
316403,26.0,1194.0
316404,807.0,1271.0
316408,81.0,1088.0


In [51]:
df_test_2['f138'].value_counts()

f138
0                         472
7250000000000000000       302
165000000000000000        289
7340000000000000000       278
1870000000000000000       247
                         ... 
2680897347654700236800      1
207237264678519996416       1
57405916001139998720        1
210504440119280009216       1
32144432821554999296        1
Name: count, Length: 31664, dtype: int64

In [52]:
df_test_2['f136'].value_counts()

f136
1.170000e+11    559
1.260000e+11    523
1.030000e+12    510
0.000000e+00    472
2.050000e+12    440
               ... 
9.612507e+10      1
6.079862e+10      1
8.487490e+08      1
4.164248e+07      1
1.109646e+07      1
Name: count, Length: 33705, dtype: int64

In [53]:
df_test_2['f135'].value_counts()

f135
0.0          470
1278755.0     40
544219.0      37
454831.0      34
469860.0      25
            ... 
3449989.0      1
449416.0       1
855450.0       1
263559.0       1
8004890.0      1
Name: count, Length: 46522, dtype: int64

In [54]:
df_test_2['f1'].value_counts()

f1
129    5769
126    5553
124    5416
130    4189
128    4122
       ... 
170      14
174      10
104      10
173       6
176       3
Name: count, Length: 73, dtype: int64

In [55]:
df_test_2.loc[df_test_2['f1']==173 , 'f85':'f86'].sum()

f85    2857.0
f86    3562.0
dtype: float64

In [56]:
df_test_2['f2'].value_counts().sort_index()

f2
1       750
2      1017
3      1318
4      2400
5        21
6      7191
7     11779
8     28722
9     28142
10    23268
11      832
Name: count, dtype: int64

In [57]:
df_test_2['f85'].value_counts()

f85
0.0       1586
1.0       1064
2.0        982
5.0        820
3.0        801
          ... 
1672.0       1
2090.0       1
2230.0       1
2305.0       1
2331.0       1
Name: count, Length: 1919, dtype: int64

In [58]:
df_test_2.loc[df_test_2['f86'] == 2331]
# 271290

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f769,f770,f771,f772,f773,f774,f775,f776,f777,f778
id,,,,,,,,,,,,,,,,,,,,,
271290,129,8,0.299884,1300,4.0,2517,950.0,1965.0,156.59,114.0,...,-16.3,2,19.4,-7.39,4.7,0.2744,0.8208,0,0,34


In [59]:
df_test_2.loc[df_test_2['f87'] == 2331]
# 281957

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f769,f770,f771,f772,f773,f774,f775,f776,f777,f778
id,,,,,,,,,,,,,,,,,,,,,
281957,154,4,0.226971,1100,7.0,9139,488.0,1979.0,125.32,128.84,...,-3.23,23,16.35,-0.07,0.38,0.2055,0.6712,0,0,1079


In [60]:
df_test_2.loc[df_test_2['f88'] == 2331]
# 258877

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f769,f770,f771,f772,f773,f774,f775,f776,f777,f778
id,,,,,,,,,,,,,,,,,,,,,
258877,121,10,0.688248,2200,4.0,78520,130.0,2088.0,119.94,122.55,...,-1.69,7,4.41,-10.67,0.25,0.2628,1.007,0,1,1079


In [61]:
df_test_2.loc[df_test_2['f89'] == 2331]

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f769,f770,f771,f772,f773,f774,f775,f776,f777,f778
id,,,,,,,,,,,,,,,,,,,,,


In [62]:
value_to_find = 2331
lst_cols_to_check = ['f85','f86','f87','f88']
lst_indices = []
for col in lst_cols_to_check:
    lst_indices += list(df_test_2.loc[df_test_2[col] == value_to_find].index)


In [63]:
df_test_2.loc[lst_indices,lst_cols_to_check].sum()

f85    2475.0
f86    3627.0
f87    3791.0
f88    3188.0
dtype: float64

In [64]:
df_test_2.loc[lst_indices,lst_cols_to_check]

,f85,f86,f87,f88
id,,,,
291658,2331.0,370.0,80.0,404.0
271290,17.0,2331.0,229.0,92.0
281957,54.0,873.0,2331.0,361.0
258877,73.0,53.0,1151.0,2331.0


In [65]:
df_test_2.loc[lst_indices,:].to_excel('data/interim/loan-default-prediction/df_test_2_2331.xlsx')

In [66]:
np_f85 = df_test_2['f85'].dropna().to_numpy()
np_f86 = df_test_2['f86'].dropna().to_numpy()

In [67]:
import numpy as np

In [68]:
np_f85.sort()
np_f86.sort()   
np.array_equal(np_f85, np_f86)

True

In [69]:
df_test.loc[[271289, 271290, 271291, 271292, 281956, 281957, 281958, 281959],['f86','f87']]

,f86,f87
id,,
271289,1249.0,1249.0
271290,2331.0,229.0
271291,20.0,20.0
271292,74.0,293.0
281956,40.0,40.0
281957,873.0,2331.0
281958,132.0,132.0
281959,2084.0,588.0


In [70]:
df_test_2.loc[[271290,  281957, ],['f85', 'f86','f87', 'f88', 'f95', 'f96', 'f97', 'f98']]

,f85,f86,f87,f88,f95,f96,f97,f98
id,,,,,,,,
271290,17.0,2331.0,229.0,92.0,318.0,709.0,3.0,287.0
281957,54.0,873.0,2331.0,361.0,261.0,321.0,118.0,15.0


In [71]:
df_test_2.loc[[271290], 'f86':'f87']

,f86,f87
id,,
271290,2331.0,229.0


In [72]:
df_test_2[df_test_2['f87'] == 873].index

Index([109577, 111501, 112399, 113121, 119010, 121211, 122001, 129569, 130061,
       130938, 130956, 132029, 134473, 138523, 139782, 140950, 144746, 147636,
       155508, 159058, 160263, 161874, 162297, 167394, 170367, 172007, 180867,
       181640, 182162, 183929, 187912, 192433, 195972, 197053, 211215, 215106,
       216751, 220700, 222492, 226086, 226598, 230055, 230341, 243669, 246311,
       248775, 252923, 255936, 256110, 256819, 258534, 262753, 265457, 270752,
       271026, 271535, 272600, 277311, 287474, 290824, 292274, 311155],
      dtype='int64', name='id')

In [73]:
df_test_2

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f769,f770,f771,f772,f773,f774,f775,f776,f777,f778
id,,,,,,,,,,,,,,,,,,,,,
105472,147,6,0.487058,1100,17.0,75506,964.0,12686.0,152.63,115.91,...,-8.71,19,3.30,-9.37,0.50,0.0539,-1.0733,0,1,1079
105475,128,7,0.038411,1300,4.0,3793,4689.0,3469.0,120.50,121.93,...,-16.83,11,0.26,-5.31,0.78,0.2826,-0.7711,0,0,394
105476,119,10,0.443620,1300,16.0,13026,2788.0,7438.0,127.00,125.98,...,-20.00,10,13.55,-0.61,0.01,0.1815,-1.0843,0,0,23
105477,158,10,0.738549,4400,4.0,80455,446.0,8547.0,153.49,121.93,...,-3.63,6,1.00,-1.76,8.22,0.2618,-0.8426,1,1,393
105482,119,9,0.770816,1600,4.0,10845,7448.0,985.0,160.77,153.97,...,-32.73,8,1.11,-2.03,3.74,0.2391,0.7694,0,0,1079
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
316403,134,10,0.643816,2600,10.0,8563,301.0,2789.0,128.75,118.20,...,-1.28,4,1.65,-0.23,0.24,0.3036,-1.1352,0,0,7
316404,124,8,0.165964,3500,3.0,3659,145.0,1990.0,127.50,113.67,...,-2.35,8,0.60,-1.38,1.16,0.2302,2.5475,0,0,37
316408,155,10,0.336004,5000,7.0,81832,238.0,7968.0,153.26,154.01,...,-5.18,40,6.19,-0.13,3.95,0.0545,-1.3494,0,0,393


In [74]:
df_test.loc[:,['f85','f86']].to_excel('data/interim/loan-default-prediction/df_test_f85_f86.xlsx')

In [75]:
print(df_test_1.loc[:,['f85','f86']].sum())
print(df_test_2.loc[:,['f85','f86']].sum())
print(df_train.loc[:,['f85','f86']].sum())


f85    38587119.0
f86    38587119.0
dtype: float64
f85    38549763.0
f86    38549763.0
dtype: float64
f85    33267437.0
f86    33267437.0
dtype: float64


In [76]:
df_train['f5'].value_counts().sort_index()

f5
1      1611
2      3842
3     14518
4     40577
7     11902
10     7854
13     2065
15     1070
16    18760
17     3272
Name: count, dtype: int64

In [77]:
df_train['f679'].value_counts().sort_index()

f679
1.0    41533
2.0    18638
3.0    30265
4.0     1189
5.0      214
6.0     3035
7.0     4204
Name: count, dtype: int64

In [78]:
df_train.shape

(105471, 770)

In [79]:
df_test.shape

(210944, 769)

In [80]:
df_train['f6'].value_counts()

f6
13699    1384
3793     1375
76831    1373
76957    1339
76973    1271
         ... 
13701       1
81415       1
84645       1
86848       1
80285       1
Name: count, Length: 1079, dtype: int64

In [81]:
df_test.loc[df_test['f6'] == 86848, ['f86', 'f87']]

,f86,f87
id,,
118949,112.0,112.0
121443,46.0,46.0
126207,726.0,42.0
143957,70.0,70.0
157093,20.0,20.0
160684,50.0,209.0
160706,54.0,908.0
162942,71.0,12.0
169080,34.0,780.0


In [85]:
df_train_numeric_head = df_train.head(1000).select_dtypes(include=np.number)

In [96]:
idx_sort = df_train_numeric_head.mean().sort_values(ascending=True).index
ser_mean = df_train_numeric_head.mean()[idx_sort]   
ser_std = df_train_numeric_head.std()[idx_sort]
df_mean_std = pd.DataFrame({'mean': ser_mean, 'std': ser_std})

In [101]:
is_equal = df_mean_std.eq(df_mean_std.shift())
all_equal = is_equal.all(axis=1)

In [105]:
ser_mean, ser_std

(f769   -8.731710e+00
 f772   -4.090320e+00
 f393   -6.277079e-01
 f384   -6.239291e-01
 f357   -6.071326e-01
             ...     
 f492    3.280967e+15
 f585    8.337051e+15
 f482    1.159593e+16
 f531    5.866165e+16
 f466    9.819674e+16
 Length: 751, dtype: float64,
 f769    1.042688e+01
 f772    5.185000e+00
 f393    1.731343e+01
 f384    6.365136e-01
 f357    7.289549e-01
             ...     
 f492    1.501874e+16
 f585    3.275578e+16
 f482    4.403577e+16
 f531    2.211856e+17
 f466    4.436691e+17
 Length: 751, dtype: float64)

In [110]:
from collections import defaultdict

In [114]:
def get_dic_mean_std_col(df, head=10_000) -> dict:
    """
    """
    df_numeric_head = df.head(head).select_dtypes(include=np.number)
    dic_m_s_col = defaultdict(list)
    ser_mean = df_numeric_head.mean()
    ser_std = df_numeric_head.std()
    for col, m, s in zip(ser_mean.index, ser_mean, ser_std):
        dic_m_s_col[(m,s)].append(col)
    return dic_m_s_col

def get_list_potentially_the_same_cols_on_mean_col(dic_m_s_col:dict) -> list:
    return [v for v in dic_m_s_col.values() if len(v) > 1]

In [129]:
def values_in_cols_the_same(df, lst_cols):
    """
    """
    first_col = df[lst_cols[0]]
    for col in lst_cols[1:]:
        if not first_col.equals(df[col]):
            return False    
    return True

In [141]:
n_rows = 10_000
dic_m_s_col = get_dic_mean_std_col(df_train, n_rows)
lst_potential = get_list_potentially_the_same_cols_on_mean_col(dic_m_s_col)
df_train_head = df_train.head(n_rows)
lst_potential_the_same_head = [lst for lst in lst_potential if values_in_cols_the_same(df_train_head, lst)]
lst_the_same = [lst for lst in lst_potential_the_same_head if values_in_cols_the_same(df_train, lst)]

In [142]:
lst_the_same == lst_potential_the_same_head

False

In [139]:
lst_potential_the_same

[['f7', 'f457', 'f488', 'f498', 'f508'],
 ['f42', 'f58'],
 ['f74', 'f345', 'f354', 'f362', 'f371', 'f379', 'f417', 'f427'],
 ['f85', 'f86', 'f87', 'f88'],
 ['f95', 'f96', 'f97', 'f98'],
 ['f105', 'f106', 'f107', 'f108'],
 ['f115', 'f116', 'f117', 'f118'],
 ['f125', 'f126', 'f127', 'f128'],
 ['f154', 'f155', 'f156', 'f157'],
 ['f164', 'f165', 'f166', 'f167'],
 ['f174', 'f175', 'f176', 'f177'],
 ['f184', 'f185', 'f186', 'f187'],
 ['f194', 'f195', 'f196', 'f197'],
 ['f224', 'f225', 'f226', 'f227'],
 ['f234', 'f235', 'f236', 'f237'],
 ['f244', 'f245', 'f246', 'f247'],
 ['f254', 'f255', 'f256', 'f257'],
 ['f264', 'f265', 'f266', 'f267'],
 ['f293', 'f294', 'f295', 'f296'],
 ['f301', 'f302', 'f303', 'f304'],
 ['f309', 'f310', 'f311', 'f312'],
 ['f317', 'f318', 'f319', 'f320'],
 ['f325', 'f326', 'f327', 'f328'],
 ['f467', 'f478'],
 ['f543', 'f553', 'f563', 'f573', 'f582'],
 ['f597', 'f599'],
 ['f722', 'f729', 'f741'],
 ['f736', 'f764']]

In [140]:
lst_potential_the_same_head

[['f7', 'f457', 'f488', 'f498', 'f508'],
 ['f36', 'f408', 'f770'],
 ['f42', 'f58'],
 ['f74', 'f345', 'f354', 'f362', 'f371', 'f379', 'f417', 'f427'],
 ['f85', 'f86', 'f87', 'f88'],
 ['f95', 'f96', 'f97', 'f98'],
 ['f105', 'f106', 'f107', 'f108'],
 ['f115', 'f116', 'f117', 'f118'],
 ['f125', 'f126', 'f127', 'f128'],
 ['f154', 'f155', 'f156', 'f157'],
 ['f164', 'f165', 'f166', 'f167'],
 ['f174', 'f175', 'f176', 'f177'],
 ['f184', 'f185', 'f186', 'f187'],
 ['f194', 'f195', 'f196', 'f197'],
 ['f224', 'f225', 'f226', 'f227'],
 ['f234', 'f235', 'f236', 'f237'],
 ['f244', 'f245', 'f246', 'f247'],
 ['f254', 'f255', 'f256', 'f257'],
 ['f264', 'f265', 'f266', 'f267'],
 ['f293', 'f294', 'f295', 'f296'],
 ['f301', 'f302', 'f303', 'f304'],
 ['f309', 'f310', 'f311', 'f312'],
 ['f317', 'f318', 'f319', 'f320'],
 ['f325', 'f326', 'f327', 'f328'],
 ['f467', 'f478'],
 ['f543', 'f553', 'f563', 'f573', 'f582'],
 ['f597', 'f599'],
 ['f722', 'f729', 'f741'],
 ['f736', 'f764']]

In [145]:
lst_potential_the_same_head

[['f7', 'f457', 'f488', 'f498', 'f508'],
 ['f36', 'f408', 'f770'],
 ['f42', 'f58'],
 ['f74', 'f345', 'f354', 'f362', 'f371', 'f379', 'f417', 'f427'],
 ['f85', 'f86', 'f87', 'f88'],
 ['f95', 'f96', 'f97', 'f98'],
 ['f105', 'f106', 'f107', 'f108'],
 ['f115', 'f116', 'f117', 'f118'],
 ['f125', 'f126', 'f127', 'f128'],
 ['f154', 'f155', 'f156', 'f157'],
 ['f164', 'f165', 'f166', 'f167'],
 ['f174', 'f175', 'f176', 'f177'],
 ['f184', 'f185', 'f186', 'f187'],
 ['f194', 'f195', 'f196', 'f197'],
 ['f224', 'f225', 'f226', 'f227'],
 ['f234', 'f235', 'f236', 'f237'],
 ['f244', 'f245', 'f246', 'f247'],
 ['f254', 'f255', 'f256', 'f257'],
 ['f264', 'f265', 'f266', 'f267'],
 ['f293', 'f294', 'f295', 'f296'],
 ['f301', 'f302', 'f303', 'f304'],
 ['f309', 'f310', 'f311', 'f312'],
 ['f317', 'f318', 'f319', 'f320'],
 ['f325', 'f326', 'f327', 'f328'],
 ['f467', 'f478'],
 ['f543', 'f553', 'f563', 'f573', 'f582'],
 ['f597', 'f599'],
 ['f722', 'f729', 'f741'],
 ['f736', 'f764']]

In [146]:
lst_the_same

[['f7', 'f457', 'f488', 'f498', 'f508'],
 ['f42', 'f58'],
 ['f74', 'f345', 'f354', 'f362', 'f371', 'f379', 'f417', 'f427'],
 ['f85', 'f86', 'f87', 'f88'],
 ['f95', 'f96', 'f97', 'f98'],
 ['f105', 'f106', 'f107', 'f108'],
 ['f115', 'f116', 'f117', 'f118'],
 ['f125', 'f126', 'f127', 'f128'],
 ['f154', 'f155', 'f156', 'f157'],
 ['f164', 'f165', 'f166', 'f167'],
 ['f174', 'f175', 'f176', 'f177'],
 ['f184', 'f185', 'f186', 'f187'],
 ['f194', 'f195', 'f196', 'f197'],
 ['f224', 'f225', 'f226', 'f227'],
 ['f234', 'f235', 'f236', 'f237'],
 ['f244', 'f245', 'f246', 'f247'],
 ['f254', 'f255', 'f256', 'f257'],
 ['f264', 'f265', 'f266', 'f267'],
 ['f293', 'f294', 'f295', 'f296'],
 ['f301', 'f302', 'f303', 'f304'],
 ['f309', 'f310', 'f311', 'f312'],
 ['f317', 'f318', 'f319', 'f320'],
 ['f325', 'f326', 'f327', 'f328'],
 ['f467', 'f478'],
 ['f543', 'f553', 'f563', 'f573', 'f582'],
 ['f597', 'f599'],
 ['f722', 'f729', 'f741'],
 ['f736', 'f764']]

In [147]:
df_train[lst_the_same[0]].head()

,f7,f457,f488,f498,f508
id,,,,,
1,7201.0,7201.0,7201.0,7201.0,7201.0
2,240.0,240.0,240.0,240.0,240.0
3,1800.0,1800.0,1800.0,1800.0,1800.0
4,7542.0,7542.0,7542.0,7542.0,7542.0
5,89.0,89.0,89.0,89.0,89.0


In [148]:
df_train_modified = df_train.copy()
for lst_col in lst_the_same:
    df_train.drop(columns=lst_col[1:], inplace=True)

In [149]:
df_train.shape, df_train_modified.shape

((105471, 689), (105471, 770))

In [150]:
n_rows = 10_000
dic_m_s_col = get_dic_mean_std_col(df_train, n_rows)
lst_potential = get_list_potentially_the_same_cols_on_mean_col(dic_m_s_col)

In [151]:
lst_potential

[['f33', 'f34', 'f35', 'f37', 'f38', 'f678', 'f700', 'f701', 'f702'],
 ['f36', 'f408', 'f770']]

In [153]:
df_train[lst_potential[1]].head()

,f36,f408,f770
id,,,
1,5,5,5
2,6,6,6
3,13,13,13
4,4,4,4
5,26,26,26


In [160]:
df_train.loc[df_train[lst_potential[1][0]] - df_train[lst_potential[1][2]] > 0, lst_potential[1]]

,f36,f408,f770
id,,,
75986,11,11,10
94550,12,12,11
98235,13,13,12


In [167]:
df_train['f778']

id
1          5
2          5
3          5
4          5
5          5
          ..
105467    93
105468    93
105469    93
105470    93
105471    93
Name: f778, Length: 105471, dtype: int64